In [1]:
from langchain_ollama import ChatOllama

local_llm = "llama3.2:3b"
llm = ChatOllama(model=local_llm, temperature=0)
llm_json_mode = ChatOllama(model=local_llm, temperature=0, format="json")

In [3]:
import os, getpass


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("TAVILY_API_KEY")
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [5]:
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=3)

In [7]:
# import os
# import json
# from langchain.text_splitter import TextSplitter
# from langchain_community.document_loaders import TextLoader
# from langchain_community.vectorstores import Chroma
# from langchain_nomic.embeddings import NomicEmbeddings
# from chromadb import PersistentClient
# from chromadb.errors import InvalidDimensionException


# class Every20LinesSplitter(TextSplitter):
#     def split_text(self, text: str) -> list[str]:
#         lines = text.split("\n")
#         return ["\n".join(lines[i : i + 10]) for i in range(0, len(lines), 10)]


# persistent_client = PersistentClient(path="path/to/persist/directory")
# collection_name = "your_collection_name"
# embedding_function = NomicEmbeddings(
#     model="nomic-embed-text-v1.5", inference_mode="local"
# )
# progress_file = "processing_progress.json"


# def load_vectorstore(read_only=False):
#     try:
#         print(f"Loading existing collection: {collection_name}")
#         return Chroma(
#             client=persistent_client,
#             collection_name=collection_name,
#             embedding_function=embedding_function,
#         )
#     except Exception as e:
#         print(f"Error loading collection: {e}")
#         return None


# def create_new_vectorstore():
#     print(f"Creating new collection: {collection_name}")
#     return Chroma(
#         client=persistent_client,
#         collection_name=collection_name,
#         embedding_function=embedding_function,
#     )


# def save_progress(processed_count):
#     with open(progress_file, "w") as f:
#         json.dump({"processed_count": processed_count}, f)


# def load_progress():
#     if os.path.exists(progress_file):
#         with open(progress_file, "r") as f:
#             return json.load(f)["processed_count"]
#     return 0


# # Try to load the existing vectorstore in read-write mode
# vecstore = load_vectorstore(read_only=False)

# if vecstore is None:
#     # If loading fails, try in read-only mode
#     vectorstore = load_vectorstore(read_only=True)
#     if vecstore is None:
#         # If read-only also fails, create a new vecstore
#         vecstore = create_new_vectorstore()

# # Load and process documents
# raw = TextLoader("processed_chat.txt").load()
# text_splitter = Every20LinesSplitter(chunk_overlap=0)
# docs = text_splitter.create_documents([raw[0].page_content])

# # Load progress
# start_index = load_progress()
# print(f"Resuming from document {start_index + 1}")

# # Process only new documents in batches of 20
# batch_size = 20
# for i in range(start_index, len(docs), batch_size):
#     batch = docs[i : i + batch_size]
#     try:
#         new_docs = [doc for doc in batch if doc.page_content not in vecstore.get()]
#         if new_docs:
#             vecstore.add_documents(new_docs)
#             processed_count = i + len(batch)
#             save_progress(processed_count)
#             print(
#                 f"Processed and saved documents {i+1} to {min(processed_count, len(docs))}"
#             )
#         else:
#             print(
#                 f"Documents {i+1} to {min(i+batch_size, len(docs))} already processed, skipping."
#             )
#     except InvalidDimensionException:
#         print(
#             "Error: Unable to add documents. The database might be in read-only mode."
#         )
#         break
#     except Exception as e:
#         print(f"Error processing batch: {e}")
#         break

# print("Processing completed.")

# # Now you can use the vectorstore for queries, etc.